# 05_kernel_ideas

公開カーネルで使われていた手法のうち，次の 2 つを試す．比べる元は 04 の候補 1（天気の前後関係を足した勾配ブースティング）で，モデルの設定は 03 から変えない．

1. 特別な日の holiday・workingday を直す（[logicalguess](https://github.com/logicalguess/kaggle-bike-sharing-demand) の code/main.py．2 位のチームと同じ名前のアカウント）
2. 年 × 季節の水準（[qinhanmin2014](https://github.com/qinhanmin2014/kaggle-bike-sharing-demand) の count_season）と，2011-01-01 からの通しの日数

最後に，04 の両方（casual と registered を別々に予測して足す）にも 1 を入れて比べる．

In [ ]:
# src/ 以下を編集したら，カーネルを再起動しなくても読み込み直す
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import PredefinedSplit, cross_val_predict, cross_val_score

from bike_sharing.cv import get_folds
from bike_sharing.data import load_data, save_submission
from bike_sharing.visualize import set_style

# print で DataFrame を出すときに，列が「...」で省略されないようにする
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
set_style()

train, test = load_data()
print("train:", train.shape, " test:", test.shape)

## 特徴量と目的変数

04 の候補 1 と同じ．あとで日付で行を選ぶので，`date`（日付の文字列）も作っておく．`date` は特徴量には使わない．

In [ ]:
for df in [train, test]:
    df["year"] = df["datetime"].dt.year
    df["month"] = df["datetime"].dt.month
    df["hour"] = df["datetime"].dt.hour
    df["dow"] = df["datetime"].dt.dayofweek
    df["date"] = df["datetime"].dt.strftime("%Y-%m-%d")

# 天気の前後関係（04 の候補 1）
weather = pd.concat([train, test]).set_index("datetime")["weather"].sort_index()
hours = pd.date_range(weather.index.min(), weather.index.max(), freq="h")
weather = weather.reindex(hours)
weather_prev = weather.shift(1)

rain_hours = []
n = 0
for w in weather:
    if w >= 3:
        n += 1
    else:
        n = 0
    rain_hours.append(n)
rain_hours = pd.Series(rain_hours, index=hours)

for df in [train, test]:
    df["weather_prev"] = df["datetime"].map(weather_prev)
    df["rain_hours"] = df["datetime"].map(rain_hours)

features = ["season", "holiday", "workingday", "weather", "temp", "atemp", "humidity", "windspeed",
            "year", "month", "hour", "dow", "weather_prev", "rain_hours"]
y = np.log1p(train["count"])

folds = get_folds(train)
block_cv = PredefinedSplit(folds)

## 04 の候補 1 の CV（比べる元）

In [ ]:
gb = HistGradientBoostingRegressor(early_stopping=False)
scores_base = -cross_val_score(gb, train[features], y, cv=block_cv, scoring="neg_root_mean_squared_error")
print("04 候補 1:", scores_base.round(4), " 平均", scores_base.mean().round(4))

oof_base = cross_val_predict(gb, train[features], y, cv=block_cv)

## 1. 特別な日

logicalguess のコードは，次の日の holiday と workingday を書き換えている．

| 日 | 何の日か | カーネルの書き換え |
|---|---|---|
| 2011-04-15，2012-04-16 | D.C. の奴隷解放記念日 | 勤務日にする |
| 2011-11-25，2012-11-23 | 感謝祭の翌日 | 休日にする |
| 2012-05-21 | 嵐（コードのコメントより） | holiday = 1 |
| 2012-06-01 | 竜巻（同じ） | holiday = 1 |
| 2012-10-30 | ハリケーン・サンディ | holiday = 1 |
| 12/24，12/26，12/31（2 年とも） | 年末 | holiday = 1．24 日と 31 日は workingday = 0 も |

train は各月の 1〜19 日なので，このうち train にあるのは 4/15，4/16，6/1 の 3 日だけ．残りは test の日で，CV では効果を測れない．

データでは今どうなっているかを見る（dow は 0 が月曜日）．

In [ ]:
special_days = ["2011-04-15", "2012-04-16", "2011-11-25", "2012-11-23", "2012-05-21", "2012-06-01", "2012-10-30",
                "2011-12-24", "2011-12-26", "2011-12-31", "2012-12-24", "2012-12-26", "2012-12-31"]

all_data = pd.concat([train.assign(data="train"), test.assign(data="test")])
days = all_data[all_data["date"].isin(special_days)]
days.groupby("date").agg(data=("data", "first"), 行数=("hour", "size"), dow=("dow", "first"),
                         holiday=("holiday", "first"), workingday=("workingday", "first"))

### 1-1. train の祝日の誤差

04 の候補 1 の out-of-fold 予測で，train の祝日を 1 日ずつ見る．

In [ ]:
err = pd.DataFrame({"date": train["date"], "holiday": train["holiday"],
                    "二乗誤差": (oof_base - y) ** 2, "予測 - 実際": oof_base - y})
holiday_err = err[err["holiday"] == 1].groupby("date").mean()
holiday_err["RMSLE"] = np.sqrt(holiday_err["二乗誤差"])

print("train 全体の RMSLE:", round(root_mean_squared_error(y, oof_base), 3))
holiday_err[["RMSLE", "予測 - 実際"]].round(3)

### 1-2. 奴隷解放記念日を勤務日にする

奴隷解放記念日（4 月 16 日）は D.C. の祝日で，2011 年は 16 日が土曜日だったので 15 日（金）に振り替えられた．連邦の祝日ではないが，この日のために確定申告の期限がずれている（2011 年は 4/18，2012 年は 4/17）．カーネルが "tax day" と書いているのはこのため．データでは holiday = 1 になっている．

直す前に，時間ごとの台数を，前の週と前の前の週の同じ曜日と比べる．

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for date in ["2011-04-01", "2011-04-08", "2011-04-15"]:
    day = train[train["date"] == date]
    plt.plot(day["hour"], day["count"], marker="o", label=date)
plt.xlabel("時間")
plt.ylabel("貸出数")
plt.title("2011-04-15（金）と前の 2 週の金曜日")
plt.legend()

plt.subplot(1, 2, 2)
for date in ["2012-04-02", "2012-04-09", "2012-04-16"]:
    day = train[train["date"] == date]
    plt.plot(day["hour"], day["count"], marker="o", label=date)
plt.xlabel("時間")
plt.ylabel("貸出数")
plt.title("2012-04-16（月）と前の 2 週の月曜日")
plt.legend()

plt.show()

In [ ]:
emancipation = train["date"].isin(["2011-04-15", "2012-04-16"])

train_a = train.copy()
train_a.loc[emancipation, "holiday"] = 0
train_a.loc[emancipation, "workingday"] = 1

scores_a = -cross_val_score(gb, train_a[features], y, cv=block_cv, scoring="neg_root_mean_squared_error")
print("奴隷解放記念日を勤務日に:", scores_a.round(4), " 平均", scores_a.mean().round(4))

In [ ]:
# 直した 2 日，ほかの祝日，ふつうの日に分けて，04 の候補 1 と比べる
oof_a = cross_val_predict(gb, train_a[features], y, cv=block_cv)

day_type = pd.Series("ふつうの日", index=train.index)
day_type[train["holiday"] == 1] = "ほかの祝日"
day_type[emancipation] = "奴隷解放記念日"
print(day_type.value_counts())

err_a = pd.DataFrame({"04 候補 1": (oof_base - y) ** 2, "直したあと": (oof_a - y) ** 2})
np.sqrt(err_a.groupby(day_type).mean()).round(3)

### 1-3. 嵐の日を休日にする

カーネルは，嵐や竜巻があった日（2012-05-21，2012-06-01）を holiday = 1 にしている．train にあるのは 2012-06-01 だけで，この日は 14 時から weather が 3 になっている．04 で雨の特徴量を足しているので，休日にする意味があるかを確かめる．1-2 の補正は入れたままにする．

In [ ]:
train.loc[train["date"] == "2012-06-01", ["hour", "weather", "rain_hours", "count"]].set_index("hour").T

In [ ]:
storm = train["date"] == "2012-06-01"

# カーネルと同じく，holiday だけ 1 にする
train_b = train_a.copy()
train_b.loc[storm, "holiday"] = 1

scores_b = -cross_val_score(gb, train_b[features], y, cv=block_cv, scoring="neg_root_mean_squared_error")
print("嵐の日も休日に:", scores_b.round(4), " 平均", scores_b.mean().round(4))

oof_b = cross_val_predict(gb, train_b[features], y, cv=block_cv)
print("2012-06-01 の RMSLE  直す前:", round(root_mean_squared_error(y[storm], oof_a[storm]), 3),
      " 直したあと:", round(root_mean_squared_error(y[storm], oof_b[storm]), 3))

### 1-4. test の日

残りは test の日なので，CV では効果を測れない．LB を見て選び直さないように，カレンダーなどの外の事実で決めておく．

休みとして扱う日は holiday = 1，workingday = 0 にする．train では holiday = 1 の日はすべて workingday = 0 なので，その組み合わせにそろえる（カーネルは 2012-12-26 を holiday = 1，workingday = 1 にしているが，この組み合わせは train に 1 行もない）．嵐の日（2012-05-21）は，1-3 で 2012-06-01 の誤差が下がらなかったので入れない．

| 日 | データ | 外の事実 |
|---|---|---|
| 2011-11-25，2012-11-23（金） | workingday = 1 | 感謝祭の翌日．連邦の祝日ではないが，休む人が多い |
| 2012-10-29（月），10-30（火） | workingday = 1．29 日は 1 行，30 日は 11 行しかない | ハリケーン・サンディで，OPM（連邦政府の人事管理局）が D.C. 周辺の官庁を 2 日とも閉めた．カーネルは 30 日だけ |
| 2012-12-24（月） | workingday = 1 | 大統領令 13633 で，連邦政府が休み |
| 2012-12-26（水），12-31（月） | workingday = 1 | 連邦の祝日ではない．年末で休む人が多い |
| 2011-12-24，12-31（土） | workingday = 0 | 土曜日なので，holiday だけ 1 になる |

その前に，効きそうかを train で見積もる．test のこれらの日は，今は勤務日として予測されている．そこで train の祝日（奴隷解放記念日を除く 11 日）を，同じように勤務日として予測してみる．学習は正しい holiday・workingday のままにして，検証する fold の祝日だけ勤務日に書き換えて予測する（test と同じ状況）．

In [ ]:
other_holiday = train_a["holiday"] == 1

train_x = train_a.copy()
train_x.loc[other_holiday, "holiday"] = 0
train_x.loc[other_holiday, "workingday"] = 1

oof_x = np.zeros(len(train))
for k in range(4):
    gb.fit(train_a.loc[folds != k, features], y[folds != k])
    oof_x[folds == k] = gb.predict(train_x.loc[folds == k, features])

compare = pd.DataFrame({"date": train["date"], "祝日として": (oof_a - y) ** 2, "勤務日として": (oof_x - y) ** 2})
compare = compare[other_holiday]
print("11 日まとめて  祝日として:", round(np.sqrt(compare["祝日として"].mean()), 3),
      " 勤務日として:", round(np.sqrt(compare["勤務日として"].mean()), 3))
np.sqrt(compare.groupby("date").mean()).round(3)

In [ ]:
day_off = ["2011-11-25", "2012-11-23", "2012-10-29", "2012-10-30",
           "2011-12-24", "2011-12-31", "2012-12-24", "2012-12-26", "2012-12-31"]
off = test["date"].isin(day_off)
print("書き換える行:", off.sum(), " そのうち workingday が 1 から 0 になる行:", (off & (test["workingday"] == 1)).sum())

test_a = test.copy()
test_a.loc[off, "holiday"] = 1
test_a.loc[off, "workingday"] = 0

## 2. 年 × 季節の水準，通しの日数

貸出数は 2011 年より 2012 年のほうが多く，同じ年の中でも季節で違う．qinhanmin2014 は，`year + season / 10`（2011.1〜2012.4 の 8 つ）ごとの count の中央値を `count_season` として足している．count から作る特徴量なので，CV では fold ごとに，学習に使う側の行だけで中央値を出す（検証する行の count を使うと，CV が甘く出る）．

もう 1 つは通しの日数（2011-01-01 から何日目か）．year と month より細かく，利用が伸びていく様子を表せるかもしれない．

比べる元は 04 の候補 1 のまま（1 の補正は入れない）．

In [ ]:
for df in [train, test]:
    df["year_season"] = df["year"] + df["season"] / 10
    df["day_index"] = (df["datetime"] - pd.Timestamp("2011-01-01")).dt.days

# train 全体で出した中央値
train.groupby("year_season")["count"].median()

In [ ]:
# count_season は fold ごとに，学習に使う側の行だけで中央値を出して足す
def cv_with_count_season(feature_list):
    scores = []
    for k in range(4):
        train_part = train[folds != k]
        valid_part = train[folds == k]
        median = train_part.groupby("year_season")["count"].median()

        X_train = train_part[feature_list].copy()
        X_valid = valid_part[feature_list].copy()
        X_train["count_season"] = train_part["year_season"].map(median)
        X_valid["count_season"] = valid_part["year_season"].map(median)

        gb.fit(X_train, y[folds != k])
        scores.append(root_mean_squared_error(y[folds == k], gb.predict(X_valid)))
    return np.array(scores)


scores_c = cv_with_count_season(features)
print("count_season を足す:", scores_c.round(4), " 平均", scores_c.mean().round(4))

In [ ]:
scores_d = -cross_val_score(gb, train[features + ["day_index"]], y, cv=block_cv, scoring="neg_root_mean_squared_error")
print("通しの日数を足す:", scores_d.round(4), " 平均", scores_d.mean().round(4))

### カーネルの特徴量には month がない

qinhanmin2014 の特徴量には year はあるが month がない．logicalguess の勾配ブースティングも同じ．自分のモデルには year と month が両方入っているので，水準はもう表せているのかもしれない．month を抜いたときと，year と month を両方抜いたときに，count_season を足して比べる．

In [ ]:
features_no_month = [f for f in features if f != "month"]
features_no_year_month = [f for f in features if f not in ["year", "month"]]

no_month = pd.DataFrame(index=["1〜5 日", "6〜10 日", "11〜14 日", "15〜19 日"])
no_month["month なし"] = -cross_val_score(gb, train[features_no_month], y, cv=block_cv,
                                         scoring="neg_root_mean_squared_error")
no_month["month なし + count_season"] = cv_with_count_season(features_no_month)
no_month["year・month なし"] = -cross_val_score(gb, train[features_no_year_month], y, cv=block_cv,
                                              scoring="neg_root_mean_squared_error")
no_month["year・month なし + count_season"] = cv_with_count_season(features_no_year_month)
no_month.loc["平均"] = no_month.mean()
no_month.round(4)

## 3. casual と registered に分けた場合（04 の両方）

04 の両方に，1-2 の補正（奴隷解放記念日を勤務日に）を入れる．

In [ ]:
y_casual = np.log1p(train["casual"])
y_registered = np.log1p(train["registered"])

# 04 の両方（補正なし）
oof_casual = cross_val_predict(gb, train[features], y_casual, cv=block_cv)
oof_registered = cross_val_predict(gb, train[features], y_registered, cv=block_cv)
oof_split = np.log1p(np.expm1(oof_casual) + np.expm1(oof_registered))

# 奴隷解放記念日を勤務日にしたもの
oof_casual_a = cross_val_predict(gb, train_a[features], y_casual, cv=block_cv)
oof_registered_a = cross_val_predict(gb, train_a[features], y_registered, cv=block_cv)
oof_split_a = np.log1p(np.expm1(oof_casual_a) + np.expm1(oof_registered_a))

scores_split = []
scores_split_a = []
for k in range(4):
    scores_split.append(root_mean_squared_error(y[folds == k], oof_split[folds == k]))
    scores_split_a.append(root_mean_squared_error(y[folds == k], oof_split_a[folds == k]))
scores_split = np.array(scores_split)
scores_split_a = np.array(scores_split_a)
print("04 両方:", scores_split.round(4), " 平均", scores_split.mean().round(4))
print("04 両方 + 奴隷解放記念日を勤務日に:", scores_split_a.round(4), " 平均", scores_split_a.mean().round(4))

In [ ]:
err_split = pd.DataFrame({"04 両方": (oof_split - y) ** 2, "直したあと": (oof_split_a - y) ** 2})
np.sqrt(err_split.groupby(day_type).mean()).round(3)

## CV のまとめ

In [ ]:
result = pd.DataFrame({
    "04 候補 1": scores_base,
    "1-2 奴隷解放記念日": scores_a,
    "1-3 + 嵐の日": scores_b,
    "2 count_season": scores_c,
    "2 通しの日数": scores_d,
    "04 両方": scores_split,
    "04 両方 + 1-2": scores_split_a,
}, index=["1〜5 日", "6〜10 日", "11〜14 日", "15〜19 日"])
result.loc["平均"] = result.mean()
result.round(4)

In [ ]:
# 比べる元との差（マイナスなら良くなった）
# 1-3 は 1-2 との差，04 両方 + 1-2 は 04 両方との差，ほかは 04 候補 1 との差
diff = pd.DataFrame(index=result.index)
diff["1-2 奴隷解放記念日"] = result["1-2 奴隷解放記念日"] - result["04 候補 1"]
diff["1-3 + 嵐の日"] = result["1-3 + 嵐の日"] - result["1-2 奴隷解放記念日"]
diff["2 count_season"] = result["2 count_season"] - result["04 候補 1"]
diff["2 通しの日数"] = result["2 通しの日数"] - result["04 候補 1"]
diff["04 両方 + 1-2"] = result["04 両方 + 1-2"] - result["04 両方"]
diff.round(4)

## 提出用のファイル

作っておくだけで，提出は Kaggle の画面からする．どれも，奴隷解放記念日を勤務日に直した train（`train_a`）で学習する．

- `05_kernel_ideas_emancipation`：04 の候補 1 のモデル
- `05_kernel_ideas_emancipation_split`：04 の両方のモデル
- `05_kernel_ideas_special_split`：2 つ目と同じモデルで，test の日を 1-4 のとおり休みにして予測する

2 つ目と 3 つ目の違いは，test の日の書き換えだけ．

In [ ]:
# 04 の候補 1 のモデル
gb.fit(train_a[features], y)
pred = np.expm1(gb.predict(test[features]))
print("0 未満になった行:", (pred < 0).sum())

save_submission(pd.DataFrame({"datetime": test["datetime"], "count": pred}), "05_kernel_ideas_emancipation")

In [ ]:
# 04 の両方のモデル
gb_casual = HistGradientBoostingRegressor(early_stopping=False)
gb_registered = HistGradientBoostingRegressor(early_stopping=False)
gb_casual.fit(train_a[features], y_casual)
gb_registered.fit(train_a[features], y_registered)

pred_split = np.expm1(gb_casual.predict(test[features])) + np.expm1(gb_registered.predict(test[features]))
print("0 未満になった行:", (pred_split < 0).sum())

save_submission(pd.DataFrame({"datetime": test["datetime"], "count": pred_split}), "05_kernel_ideas_emancipation_split")

In [ ]:
# test の日を休みにして予測する（モデルは上と同じ）
pred_special = np.expm1(gb_casual.predict(test_a[features])) + np.expm1(gb_registered.predict(test_a[features]))
print("0 未満になった行:", (pred_special < 0).sum())

# 書き換えた日の 1 日の合計台数が，どれくらい変わったか
change = pd.DataFrame({"date": test["date"], "書き換え前": pred_split, "書き換え後": pred_special})
print(change[off].groupby("date").sum().round(0))

save_submission(pd.DataFrame({"datetime": test["datetime"], "count": pred_special}), "05_kernel_ideas_special_split")